# Neural Network Fundamentals
**CubicNova Data Science Roadmap · Module 10.01**

This notebook introduces neurons, activation functions, forward propagation, gradient descent, a NumPy neural network, and a small PyTorch classifier. Run it from top to bottom.

**Learning pattern:** understand the math → run a small example → change one thing → interpret the result.

## 0. Imports and reproducibility

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("NumPy:", np.__version__)

## 1. A single neuron: weighted sum plus bias

In [ ]:
x = np.array([2.0, -1.0])
w = np.array([0.5, 3.0])
b = 1.0

z = np.dot(x, w) + b
print("Weighted sum plus bias:", z)

The neuron first calculates `z = x · w + b`. An activation function transforms this value. Weights control input influence; the bias shifts the response.

## 2. Activation functions

In [ ]:
def sigmoid(x):
    x = np.asarray(x, dtype=float)
    # Clipping keeps this teaching example away from overflow.
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def tanh(x):
    return np.tanh(x)

def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=-1, keepdims=True)

values = np.linspace(-5, 5, 200)
plt.figure(figsize=(8, 4))
plt.plot(values, sigmoid(values), label="sigmoid")
plt.plot(values, tanh(values), label="tanh")
plt.plot(values, relu(values), label="ReLU")
plt.title("Common activation functions")
plt.xlabel("z")
plt.ylabel("activation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Softmax probabilities:", softmax([2.0, 1.0, 0.1]))
print("Probability sum:", softmax([2.0, 1.0, 0.1]).sum())

Sigmoid maps values to (0, 1), tanh to (-1, 1), and ReLU clips negative values to zero. Softmax converts a vector of logits into values that sum to one.

## 3. A perceptron for the AND gate

In [ ]:
def perceptron(x, weights, bias):
    return int(np.dot(np.asarray(x), np.asarray(weights)) + bias >= 0)

weights = np.array([1.0, 1.0])
bias = -1.5
for x in ([0, 0], [0, 1], [1, 0], [1, 1]):
    print(f"{x} -> {perceptron(x, weights, bias)}")

A perceptron draws a linear decision boundary. It can represent AND and OR, but one perceptron cannot represent XOR because XOR is not linearly separable.

## 4. A dense layer forward pass

In [ ]:
# Batch of 2 samples, 3 features each
X = np.array([[1.0, 2.0, 0.5],
              [0.0, -1.0, 3.0]])
# 3 input features -> 2 neurons
W = np.array([[0.2, -0.4],
              [0.7,  0.1],
              [-0.3, 0.8]])
b = np.array([0.1, -0.2])

Z = X @ W + b
A = relu(Z)
print("Z shape:", Z.shape)
print("Activated output:\n", A)

For a batch, matrix multiplication computes all examples together. With `X.shape == (batch_size, input_dim)`, `W.shape == (input_dim, output_dim)`, and `b.shape == (output_dim,)`, the output shape is `(batch_size, output_dim)`.

## 5. Gradient descent on a simple function

In [ ]:
# Minimize f(w) = (w - 3)^2; derivative is 2(w - 3)
w = -4.0
learning_rate = 0.1
history = []

for step in range(40):
    loss = (w - 3) ** 2
    grad = 2 * (w - 3)
    history.append(loss)
    w -= learning_rate * grad

print("Final w:", round(w, 4))
print("Final loss:", round((w - 3) ** 2, 8))

plt.figure(figsize=(7, 3))
plt.plot(history)
plt.xlabel("Update")
plt.ylabel("Loss")
plt.title("Gradient descent reduces loss")
plt.grid(True, alpha=0.3)
plt.show()

Gradient descent changes parameters in the negative-gradient direction. The learning rate controls the size of each update.

## 6. Train a tiny neural network on XOR (NumPy)

This compact implementation uses one hidden layer with sigmoid activations and mean squared error. It is for learning the mechanics; production training should use numerically stable loss implementations.

In [ ]:
# XOR dataset
X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = np.array([[0.], [1.], [1.], [0.]])

def sigmoid_np(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

rng = np.random.default_rng(7)
W1 = rng.normal(0, 1, (2, 4))
b1 = np.zeros((1, 4))
W2 = rng.normal(0, 1, (4, 1))
b2 = np.zeros((1, 1))

lr = 0.8
loss_history = []
epochs = 10000

for epoch in range(epochs):
    # Forward pass
    z1 = X @ W1 + b1
    a1 = sigmoid_np(z1)
    z2 = a1 @ W2 + b2
    y_hat = sigmoid_np(z2)

    # Mean squared error and gradients (full batch)
    loss = np.mean((y_hat - y) ** 2)
    loss_history.append(loss)

    # Backpropagation: dL/da2, sigmoid derivative, then chain rule
    d_z2 = (2 / y.shape[0]) * (y_hat - y) * y_hat * (1 - y_hat)
    dW2 = a1.T @ d_z2
    db2 = np.sum(d_z2, axis=0, keepdims=True)

    d_a1 = d_z2 @ W2.T
    d_z1 = d_a1 * a1 * (1 - a1)
    dW1 = X.T @ d_z1
    db1_grad = np.sum(d_z1, axis=0, keepdims=True)

    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1_grad

print("Final loss:", round(loss_history[-1], 6))
print("Predictions:")
print(np.round(y_hat, 3))
print("Predicted classes:")
print((y_hat >= 0.5).astype(int))

plt.figure(figsize=(7, 3))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("XOR network training loss")
plt.grid(True, alpha=0.3)
plt.show()

**Observe:** the hidden layer gives the model enough nonlinear capacity to represent XOR. Initialization and hyperparameters can affect convergence, so rerun with different seeds or learning rates and compare.

## 7. A small classifier with PyTorch

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

torch.manual_seed(42)
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # fit only on training data
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True
)

model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)  # logits; no sigmoid here
)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

train_losses = []
for epoch in range(100):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    train_losses.append(total_loss / len(train_loader.dataset))

model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    test_probs = torch.sigmoid(test_logits).numpy().ravel()
    test_pred = (test_probs >= 0.5).astype(int)

print("Test accuracy:", round((test_pred == y_test).mean(), 3))
print(classification_report(y_test, test_pred, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_test, test_pred))

plt.figure(figsize=(7, 3))
plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training BCE loss")
plt.title("PyTorch MLP training loss")
plt.grid(True, alpha=0.3)
plt.show()

**Important:** `BCEWithLogitsLoss` expects raw logits and combines sigmoid with binary cross-entropy in a numerically stable way. The test set is only used for evaluation, not gradient updates.

## 8. Experiment checklist

Run at least three controlled experiments:

1. Learning rate: `0.001`, `0.01`, `0.1`.
2. Hidden width: `4`, `8`, `16`.
3. Epoch count: `20`, `50`, `100`.

For each run, record the setting, final training loss, test accuracy, and any signs of instability or overfitting. Change one variable at a time. Do not select a model based on test performance repeatedly; use validation data for model selection.

## Key takeaways

- A neuron computes a weighted sum, adds bias, and applies an activation.
- Nonlinear activations let multilayer networks represent nonlinear relationships.
- Forward propagation computes predictions; backpropagation computes gradients.
- An optimizer uses gradients to update parameters.
- Always evaluate on held-out data and compare against a simple baseline.